# vLLM on Kaggle T4: the same sweep as the Mac

Before running: **Settings → Accelerator → GPU T4 x2**, **Internet → On** (needs a phone-verified account).

Runs, one server at a time:

| label | model | GPUs | why |
|---|---|---|---|
| `t4-3b-fp16` | Qwen2.5-3B-Instruct, FP16 | 1 | compare with the Mac `3b-p4` run |
| `t4-7b-awq` | Qwen2.5-7B-Instruct-AWQ, 4-bit | 1 | 7B FP16 (15.2 GB) does not fit one T4 |
| `t4x2-7b-fp16-tp2` | Qwen2.5-7B-Instruct, FP16 | 2 | optional: tensor parallel across both T4s |

Each run saves the client sweep table, vLLM's own `/metrics`, the server log, and `nvidia-smi` sampled every second.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

Install vLLM (5–10 minutes). T4 is compute capability 7.5: FP16 works, BF16 and FP8 do not.

In [ ]:
!pip install -q vllm
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| capability', torch.cuda.get_device_capability(0))"

Clone the harness. An editable install keeps results inside the clone.

In [ ]:
!git clone -q https://github.com/ravi-tej-prog/ai-infra-lab.git /kaggle/working/ai-infra-lab
!pip install -q -e /kaggle/working/ai-infra-lab/qwen-inference-platform
%cd /kaggle/working/ai-infra-lab/qwen-inference-platform

In [ ]:
import os, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

ROOT = Path("/kaggle/working/ai-infra-lab/qwen-inference-platform")
SWEEPS = ROOT / "benchmark" / "results" / "sweeps"
PORT = 8000


def start_server(model, args, gpus, log_path):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus)
    cmd = ["vllm", "serve", model, "--port", str(PORT), *args]
    print(" ".join(cmd))
    log = open(log_path, "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)


def wait_ready(proc, log_path, timeout=1200):
    started = time.time()
    while time.time() - started < timeout:
        if proc.poll() is not None:
            print(Path(log_path).read_text()[-4000:])
            raise RuntimeError("vLLM exited before it was ready")
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2)
            print(f"ready after {time.time() - started:.0f}s")
            return
        except Exception:
            time.sleep(5)
    raise TimeoutError("vLLM not ready")


def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        proc.kill()
    time.sleep(10)  # let the GPU memory free before the next server
    subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv"])


def run(label, model, server_args, gpus="0", config="benchmark/configs/vllm-t4-sweep.yaml"):
    out = SWEEPS / label
    out.mkdir(parents=True, exist_ok=True)
    server_log = out / "vllm-server.log"
    proc = start_server(model, server_args, gpus, server_log)
    sampler = None
    try:
        wait_ready(proc, server_log)
        sampler = subprocess.Popen(
            ["nvidia-smi",
             "--query-gpu=timestamp,index,utilization.gpu,memory.used,power.draw,clocks.sm",
             "--format=csv", "-l", "1"],
            stdout=open(out / "gpu-samples.csv", "w"),
        )
        subprocess.run(
            [sys.executable, "-m", "qwen_platform", "sweep", "--config", config,
             "--label", label, "--model", model, "--mark-nvidia-baseline"],
            cwd=ROOT, check=False,
        )
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/metrics", timeout=10) as response:
            (out / "vllm-metrics.prom").write_bytes(response.read())
    finally:
        if sampler:
            sampler.terminate()
        stop_server(proc)
    print((out / "sweep.md").read_text() if (out / "sweep.md").exists() else "no table written")

## Run 1: 3B FP16, one T4

`--dtype half` because T4 has no BF16. `--max-model-len 4096` keeps KV cache room. Prefix caching is on by default in vLLM; the sweep's random prompt tags stop it from hiding prefill.

In [ ]:
run(
    "t4-3b-fp16",
    "Qwen/Qwen2.5-3B-Instruct",
    ["--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
)

## Run 2: 7B AWQ (4-bit), one T4

In [ ]:
run(
    "t4-7b-awq",
    "Qwen/Qwen2.5-7B-Instruct-AWQ",
    ["--quantization", "awq", "--dtype", "half", "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
)

## Run 3 (optional): 7B FP16, tensor parallel across both T4s

In [ ]:
run(
    "t4x2-7b-fp16-tp2",
    "Qwen/Qwen2.5-7B-Instruct",
    ["--dtype", "half", "--tensor-parallel-size", "2", "--max-model-len", "4096", "--gpu-memory-utilization", "0.90"],
    gpus="0,1",
)

## Package results: download `results.zip` from the Output panel

In [ ]:
shutil.make_archive("/kaggle/working/results", "zip", SWEEPS)
for table in sorted(SWEEPS.glob("*/sweep.md")):
    print(table.read_text())